# Transformery i Mechanizm Samoatencji (Self-Attention)

### Dlaczego Transformery wyparły LSTM?
* **LSTM:** Czyta tekst słowo po słowie, od lewej do prawej. Uczy się wolno i często zapomina początek długiego zdania.
* **Transformer:** Patrzy na **cały tekst jednocześnie**. Przetwarza wszystkie słowa równolegle za pomocą jednego gigantycznego mnożenia macierzy.

### Skoro czyta wszystko naraz, to jak rozumie kontekst? (Samoatencja)
Zamiast przypisywać słowu stałe znaczenie, Transformer zmusza każde słowo do "spojrzenia" na wszystkie inne słowa w zdaniu. Służą do tego 3 wektory dla każdego słowa (jak w wyszukiwarce internetowej):

1. **Query (Zapytanie - Q):** Czego szukam w tym zdaniu, żeby zrozumieć mój kontekst?
2. **Key (Klucz - K):** Kim jestem? (Etykieta słowa).
3. **Value (Wartość - V):** Moje faktyczne znaczenie.

**"Matematyka" transformerów:**
1. Mnożymy *Zapytanie (Q)* słowa "Zamek" przez *Klucze (K)* wszystkich innych słów (np. "kurtka", "rycerz"). 
2. Wynikiem jest **Uwaga (Attention Score)** - procentowo określone, jak bardzo słowa do siebie pasują.
3. Mnożymy tę Uwagę przez *Wartości (V)* dopasowanych słów, by zaktualizować znaczenie słowa "Zamek".

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
class SimpleSelfAttention(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.embed_size = embed_size
        
        # Tworzymy 3 niezależne warstwy liniowe. 
        # To one nauczą się tworzyć z każdego słowa odpowiednie wektory Q, K i V.
        self.queries = nn.Linear(embed_size, embed_size, bias=False)
        self.keys    = nn.Linear(embed_size, embed_size, bias=False)
        self.values  = nn.Linear(embed_size, embed_size, bias=False)
        
    def forward(self, x):
        # Parametr 'x' to nasze zdanie. Ma 3 wymiary: 
        # [Wielkość_paczki (Batch), Ilość_słów, Wymiar_wektora_słowa (Embed_size)]
        
        # 1. TWORZENIE Q, K, V
        # Transformer robi to dla wszystkich słów w zdaniu JEDNOCZEŚNIE!
        Q = self.queries(x)
        K = self.keys(x)
        V = self.values(x)
        
        # 2. JAK BARDZO SŁOWA DO SIEBIE PASUJĄ? (Q * K)
        # Używamy torch.bmm (Batch Matrix Multiplication) - to hurtowe mnożenie macierzy.
        # Transponujemy klucze (zamieniamy wiersze z kolumnami), żeby wymiary się zgadzały przy mnożeniu.
        # Dzielimy przez pierwiastek z 'embed_size', co jest matematycznym trikiem 
        # zapobiegającym tzw. 'eksplodującym gradientom' przed funkcją Softmax.
        scores = torch.bmm(Q, K.transpose(1, 2)) / (self.embed_size ** 0.5)
        
        # 3. ZAMIANA WYNIKÓW NA PROCENTY (Softmax)
        # Softmax sprawia, że uwaga, którą dane słowo poświęca reszcie zdania, sumuje się do 1 (100%).
        attention_weights = F.softmax(scores, dim=-1)
        
        # 4. WCHŁANIANIE KONTEKSTU (Uwaga * V)
        # Każde słowo łączy się ze znaczeniami innych słów, proporcjonalnie do siły dopasowania.
        out = torch.bmm(attention_weights, V)
        
        # Zwracamy zaktualizowane słowa oraz tabelę z wynikami uwagi (do wizualizacji)
        return out, attention_weights

print("✅ Moduł Samoatencji gotowy! Przejdź do następnej komórki, aby go przetestować.")

✅ Moduł Samoatencji gotowy! Przejdź do następnej komórki, aby go przetestować.


### Testowanie modułu 
Symulujemy "zdanie", które składa się z **4 słów**. Każde słowo będzie reprezentowane przez wektor liczb o rozmiarze **8 wymiarów**. 

Najciekawszym elementem będzie wyświetlenie `attention_weights`. Zobaczycie macierz 4x4.
Każdy wiersz to jedno słowo, które "patrzy" na pozostałe. Liczby w wierszu muszą sumować się do 1.0 (100% uwagi rozdzielone na całe zdanie).

In [7]:
# Ustawiamy ziarno losowości, żeby wyniki u każdego studenta były powtarzalne
torch.manual_seed(42)

# Symulujemy zdanie: 1 paczka (batch), 4 słowa, każde o rozmiarze 8 liczb.
dummy_sentence = torch.rand(1, 4, 8) 

# Tworzymy moduł
attention_layer = SimpleSelfAttention(embed_size=8)

# Wykonujemy operację w przód (Forward pass)
output, weights = attention_layer(dummy_sentence)

print("--- WYMIARY ---")
print(f"Kształt wejścia: {dummy_sentence.shape}  -> [Paczka, Słowa, Wymiar_słowa]")
print(f"Kształt wyjścia: {output.shape}  -> Taki sam jak wejście! Znaczenia zostały zaktualizowane.")
print(f"Kształt macierzy Uwagi (Attention): {weights.shape} -> [Paczka, Słowa_patrzące, Słowa_obserwowane]")

print("\n--- MACIERZ UWAGI (ATTENTION WEIGHTS) DLA NASZEGO ZDANIA ---")
# Wyświetlamy tylko pierwszą macierz z naszej paczki
attention_matrix = weights[0].detach().numpy()
print(attention_matrix)

print("\n(Sprawdzenie matematyczne: Zobacz, że sumując liczby poziomo (w wierszach), otrzymasz dokładnie 1.0!)")

--- WYMIARY ---
Kształt wejścia: torch.Size([1, 4, 8])  -> [Paczka, Słowa, Wymiar_słowa]
Kształt wyjścia: torch.Size([1, 4, 8])  -> Taki sam jak wejście! Znaczenia zostały zaktualizowane.
Kształt macierzy Uwagi (Attention): torch.Size([1, 4, 4]) -> [Paczka, Słowa_patrzące, Słowa_obserwowane]

--- MACIERZ UWAGI (ATTENTION WEIGHTS) DLA NASZEGO ZDANIA ---
[[0.249833   0.23417084 0.26944214 0.24655405]
 [0.24209924 0.24138552 0.2593553  0.25715995]
 [0.24175669 0.23799983 0.26220173 0.25804183]
 [0.25558347 0.24375273 0.25706974 0.2435941 ]]

(Sprawdzenie matematyczne: Zobacz, że sumując liczby poziomo (w wierszach), otrzymasz dokładnie 1.0!)


### PRZYKŁAD: Prawdziwy Transformer z Hugging Face
Napisaną wyżej mechanikę łączy się w tzw. Wielogłowową Uwagę (Multi-Head Attention) i układa z niej gigantyczne bloki. Zamiast budować to od zera przez najbliższe tygodnie, weźmiemy "gotowca" – model językowy wytrenowany na milionach tekstów z polskiego internetu (polską wersję GPT-2).

Wystarczą 3 linijki kodu, by z niego skorzystać. 

In [10]:
# Instalacja biblioteki transformers od firmy Hugging Face (odkomentuj poniższą linię, jeśli uruchamiasz to po raz pierwszy)
!pip install transformers -q

from transformers import pipeline

print("Pobieranie polskiego modelu GPT-2 (to może zająć chwilkę przy pierwszym uruchomieniu)...")
# Używamy pipeline 'text-generation' i polskiego modelu "papuGaPT2"
generator = pipeline('text-generation', model='sdadas/polish-gpt2-small')

print("\nModel gotowy!")
# Podajemy modelowi początek zdania, a Transformer użyje Samoatencji, by dopisać ciąg dalszy
prompt = "Sztuczna inteligencja potrafi już bardzo dużo, jednak wciąż nie potrafi"

# Generujemy odpowiedź (max_length to ilość słów generowanych przez model)
wynik = generator(prompt, max_length=40, num_return_sequences=1)

print("\n================ WYNIK MODELU ================\n")
print(wynik[0]['generated_text'])
print("\n==============================================")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Pobieranie polskiego modelu GPT-2 (to może zająć chwilkę przy pierwszym uruchomieniu)...


config.json:   0%|          | 0.00/835 [00:00<?, ?B/s]

C:\Users\krzys\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\krzys\.cache\huggingface\hub\models--sdadas--polish-gpt2-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/554M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model gotowy!


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



================ WYNIK MODELU ================

Sztuczna inteligencja potrafi już bardzo dużo, jednak wciąż nie potrafi dobrze ocenić otaczającej rzeczywistości. Mimo to ma do dyspozycji cały arsenał narzędzi, które pozwalają na tworzenie programów, które pozwalają na tworzenie programów, które pozwalają na tworzenie programów, które pozwalają na tworzenie programów, które pozwalają na tworzenie programów, które pozwalają na tworzenie programów. Nie ma natomiast żadnej możliwości, by to wszystko samo z siebie wytworzyło. Mając na uwadze powyższe, chciałbym zapytać, czy Ministerstwo Edukacji Narodowej zna możliwość wprowadzenia do szkół na terenie całego kraju obowiązkowych zajęć z matematyki? Z poważaniem Poseł Adam Kwiatkowski Warszawa, dnia 15 września 2012 r.

